# Event Sourcing for Agentic AI: Architecting Deterministic, Auditable, and Replayable LLM Systems
**Author:** Alan Arantes (Enterprise & System Architect)  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AlanArantes/AI-Experiments/blob/main/event-sourcing-for-agentic-ai-deterministic-auditable-replayable/event_sourcing_agentic_ai_colab_notebook.ipynb)

---

### Architectural Paradigm Shift: The Log is the Source of Truth
Traditional LLM application architectures update state in-place (e.g. overwriting session dictionaries or memory records). When an autonomous agent encounters an unexpected failure, takes an unauthorized detour, or executes a hallucinated tool call, destructive state mutations make root-cause analysis impossible.

By adopting **Event Sourcing**, we treat the **append-only event log as the single source of truth**. Every cognitive intention, tool invocation, policy check, and external observation is recorded as an immutable domain event.

The agent's current state is a pure deterministic projection:
$$\text{State}_t = \text{fold}(\text{State}_0, [\text{Event}_1, \dots, \text{Event}_t])$$

This guarantees:
1. **Deterministic Replay:** Replaying exact scenarios offline with recorded tool stubs.
2. **Forensic Auditability:** Cryptographic, hash-chained provenance for compliance.
3. **Time-Travel Debugging:** Reconstructing agent state at any arbitrary step $k$.
4. **Counterfactual 'What-If' Branching:** Forking an execution stream to test alternate model parameters without mutating historical truth.


## 1. Environment Setup
Install required dependencies: Pydantic v2 (for contract models), Requests, PyYAML, and Pytest.

In [ ]:
# Install dependencies for Google Colab
!pip install -q "pydantic>=2.0" requests pyyaml pytest

import os
import json
import uuid
import hashlib
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable, Dict, List, Optional
from pydantic import BaseModel, ConfigDict, Field

print("Environment initialized successfully.")


## 2. Domain Events & Immutable Contracts
Events represent immutable facts that *have already happened* in the agent's lifecycle.

In [ ]:
# events.py
class EventType(str, Enum):
    SESSION_STARTED = "SessionStarted"
    PROMPT_RECEIVED = "PromptReceived"
    COGNITION_PLANNED = "CognitionPlanned"
    TOOL_EXECUTION_REQUESTED = "ToolExecutionRequested"
    POLICY_EVALUATED = "PolicyEvaluated"
    TOOL_EXECUTION_COMPLETED = "ToolExecutionCompleted"
    OUTPUT_GENERATED = "OutputGenerated"
    SESSION_HALTED = "SessionHalted"
    CHECKPOINT_CREATED = "CheckpointCreated"


class BaseAgentEvent(BaseModel):
    event_id: str = Field(default_factory=lambda: str(uuid.uuid4()))
    session_id: str = Field(description="Unique identifier of the agent session")
    sequence_number: int = Field(description="Strict monotonically increasing sequence number")
    timestamp: str = Field(default_factory=lambda: datetime.now(timezone.utc).isoformat())
    event_type: EventType = Field(description="Domain event discriminator")
    payload: Dict[str, Any] = Field(default_factory=dict)
    metadata: Dict[str, Any] = Field(default_factory=dict)
    causation_id: Optional[str] = None
    correlation_id: str = Field(default_factory=lambda: str(uuid.uuid4()))

    model_config = ConfigDict(frozen=True)  # Enforce immutability


# Sample event creation
sample_evt = BaseAgentEvent(
    session_id="demo-session",
    sequence_number=1,
    event_type=EventType.SESSION_STARTED,
    payload={"role": "compliance_officer", "tenant": "EnterpriseBank"}
)
print("Sample Immutable Event Schema:")
print(sample_evt.model_dump_json(indent=2))


## 3. Append-Only Event Store with Hash-Chained Provenance
The `EventStore` provides:
- Strict monotonic sequence ordering (rejects out-of-order writes).
- Cryptographic SHA256 hash chaining (detects any retroactive tampering).

In [ ]:
# event_store.py
class EventStoreConcurrencyError(Exception):
    pass

class EventStoreTamperError(Exception):
    pass


class EventStore:
    """Append-only event store with optimistic concurrency and hash-chained integrity."""

    def __init__(self):
        self._streams: Dict[str, List[BaseAgentEvent]] = {}

    def _compute_hash(self, prev_hash: str, event: BaseAgentEvent) -> str:
        hasher = hashlib.sha256()
        payload_str = json.dumps(event.payload, sort_keys=True)
        token = f"{prev_hash}|{event.session_id}|{event.sequence_number}|{event.event_type.value}|{payload_str}"
        hasher.update(token.encode("utf-8"))
        return hasher.hexdigest()

    def append(self, event: BaseAgentEvent) -> BaseAgentEvent:
        session_id = event.session_id
        if session_id not in self._streams:
            self._streams[session_id] = []

        stream = self._streams[session_id]
        expected_seq = len(stream) + 1

        if event.sequence_number != expected_seq:
            raise EventStoreConcurrencyError(
                f"Concurrency error: expected sequence {expected_seq}, got {event.sequence_number}"
            )

        prev_hash = stream[-1].metadata.get("hash", "GENESIS") if stream else "GENESIS"
        current_hash = self._compute_hash(prev_hash, event)

        updated_meta = dict(event.metadata)
        updated_meta["prev_hash"] = prev_hash
        updated_meta["hash"] = current_hash

        persisted = event.model_copy(update={"metadata": updated_meta})
        stream.append(persisted)
        return persisted

    def get_stream(self, session_id: str, from_sequence: int = 1, to_sequence: Optional[int] = None) -> List[BaseAgentEvent]:
        if session_id not in self._streams:
            return []
        stream = self._streams[session_id]
        return [e for e in stream if e.sequence_number >= from_sequence and (to_sequence is None or e.sequence_number <= to_sequence)]

    def verify_stream_integrity(self, session_id: str) -> bool:
        stream = self.get_stream(session_id)
        prev_hash = "GENESIS"
        for e in stream:
            computed = self._compute_hash(prev_hash, e)
            if e.metadata.get("hash") != computed:
                raise EventStoreTamperError(f"Integrity check failed at sequence {e.sequence_number}")
            prev_hash = computed
        return True

print("EventStore ready.")


## 4. State as a Pure Projection (Fold Function)
The agent's state is computed by reducing over the sequence of domain events. It is a pure function: given the same events, it produces identical state.

In [ ]:
# agent_state.py
class AgentState(BaseModel):
    session_id: str
    status: str = "INITIALIZED"
    current_sequence: int = 0
    user_query: Optional[str] = None
    plan: Optional[str] = None
    conversation_history: List[Dict[str, str]] = Field(default_factory=list)
    working_memory: Dict[str, Any] = Field(default_factory=dict)
    tools_invoked: List[Dict[str, Any]] = Field(default_factory=list)
    final_output: Optional[str] = None
    error_message: Optional[str] = None
    policy_violations: int = 0


def apply_event(state: AgentState, event: BaseAgentEvent) -> AgentState:
    new_state = state.model_copy(deep=True)
    new_state.current_sequence = event.sequence_number

    match event.event_type:
        case EventType.SESSION_STARTED:
            new_state.status = "RUNNING"
            new_state.working_memory.update(event.payload.get("initial_context", {}))

        case EventType.PROMPT_RECEIVED:
            prompt = event.payload.get("prompt", "")
            new_state.user_query = prompt
            new_state.conversation_history.append({"role": "user", "content": prompt})
            new_state.status = "PROCESSING"

        case EventType.COGNITION_PLANNED:
            plan = event.payload.get("plan", "")
            new_state.plan = plan
            new_state.conversation_history.append({"role": "assistant", "thought": plan})

        case EventType.TOOL_EXECUTION_REQUESTED:
            new_state.tools_invoked.append({
                "tool": event.payload.get("tool"), "args": event.payload.get("args"), "status": "PENDING"
            })
            new_state.status = "AWAITING_TOOL"

        case EventType.POLICY_EVALUATED:
            if not event.payload.get("allowed", True):
                new_state.policy_violations += 1
                if new_state.tools_invoked:
                    new_state.tools_invoked[-1]["status"] = "BLOCKED"
                    new_state.tools_invoked[-1]["block_reason"] = event.payload.get("reason")

        case EventType.TOOL_EXECUTION_COMPLETED:
            res = event.payload.get("result")
            tool_name = event.payload.get("tool")
            if new_state.tools_invoked:
                new_state.tools_invoked[-1]["status"] = "COMPLETED"
                new_state.tools_invoked[-1]["result"] = res
            new_state.working_memory[f"tool_result_{tool_name}"] = res
            new_state.conversation_history.append({"role": "system", "observation": str(res)})
            new_state.status = "PROCESSING"

        case EventType.OUTPUT_GENERATED:
            output = event.payload.get("output", "")
            new_state.final_output = output
            new_state.conversation_history.append({"role": "assistant", "content": output})
            new_state.status = "COMPLETED"

        case EventType.SESSION_HALTED:
            new_state.status = "HALTED"
            new_state.error_message = event.payload.get("reason", "Halted")

    return new_state


def reconstruct_state(events: List[BaseAgentEvent], initial_state: Optional[AgentState] = None) -> AgentState:
    state = initial_state or AgentState(session_id=events[0].session_id)
    for event in events:
        state = apply_event(state, event)
    return state

print("State Projection Reducer ready.")


## 5. Event-Sourced Agent Runtime Orchestrator
The `EventSourcedAgent` executes cognitive tasks while emitting discrete domain events into the store.

In [ ]:
# agent_orchestrator.py
class EnterpriseBackendTools:
    @staticmethod
    def get_order_details(order_id: str) -> Dict[str, Any]:
        return {"order_id": order_id, "item": "Cloud Cluster GPU", "amount": 1200.0, "status": "Active"}

    @staticmethod
    def process_refund(order_id: str, amount: float) -> Dict[str, Any]:
        return {"order_id": order_id, "amount": amount, "status": "REFUND_COMPLETED", "tx_id": "TX-7788"}


class EventSourcedAgent:
    def __init__(self, session_id: str, event_store: EventStore, tools: Optional[Dict[str, Callable]] = None):
        self.session_id = session_id
        self.event_store = event_store
        self.tools = tools or {
            "get_order_details": EnterpriseBackendTools.get_order_details,
            "process_refund": EnterpriseBackendTools.process_refund
        }

    def _emit(self, event_type: EventType, payload: Dict[str, Any], causation_id: Optional[str] = None) -> BaseAgentEvent:
        stream = self.event_store.get_stream(self.session_id)
        event = BaseAgentEvent(
            session_id=self.session_id,
            sequence_number=len(stream) + 1,
            event_type=event_type,
            payload=payload,
            causation_id=causation_id
        )
        return self.event_store.append(event)

    def run(self, prompt: str, is_read_only: bool = True) -> AgentState:
        stream = self.event_store.get_stream(self.session_id)
        last_id = None
        if not stream:
            e_init = self._emit(EventType.SESSION_STARTED, {"tenant": "AcmeCorp"})
            last_id = e_init.event_id

        e_prompt = self._emit(EventType.PROMPT_RECEIVED, {"prompt": prompt}, causation_id=last_id)
        last_id = e_prompt.event_id

        # Infer tool intent
        p_low = prompt.lower()
        if "refund" in p_low:
            plan = "Execute financial refund for order ORD-101"
            tool_call = {"tool": "process_refund", "args": {"order_id": "ORD-101", "amount": 1200.0}}
        else:
            plan = "Retrieve order records for ORD-101"
            tool_call = {"tool": "get_order_details", "args": {"order_id": "ORD-101"}}

        e_cog = self._emit(EventType.COGNITION_PLANNED, {"plan": plan}, causation_id=last_id)
        last_id = e_cog.event_id

        # Tool execution with policy gate
        tool_name = tool_call["tool"]
        tool_args = tool_call["args"]
        e_req = self._emit(EventType.TOOL_EXECUTION_REQUESTED, {"tool": tool_name, "args": tool_args}, causation_id=last_id)
        last_id = e_req.event_id

        # Policy check: Block write action during read-only sessions
        if is_read_only and tool_name == "process_refund":
            e_pol = self._emit(EventType.POLICY_EVALUATED, {"allowed": False, "reason": "Financial write blocked in read-only mode"}, causation_id=last_id)
            self._emit(EventType.SESSION_HALTED, {"reason": "Unauthorized policy violation"}, causation_id=e_pol.event_id)
            return reconstruct_state(self.event_store.get_stream(self.session_id))

        e_pol = self._emit(EventType.POLICY_EVALUATED, {"allowed": True, "reason": "Policy allowed"}, causation_id=last_id)
        last_id = e_pol.event_id

        res = self.tools[tool_name](**tool_args)
        e_res = self._emit(EventType.TOOL_EXECUTION_COMPLETED, {"tool": tool_name, "result": res}, causation_id=last_id)
        last_id = e_res.event_id

        out_text = f"Action completed for {tool_name}: {res}"
        e_out = self._emit(EventType.OUTPUT_GENERATED, {"output": out_text}, causation_id=last_id)
        self._emit(EventType.CHECKPOINT_CREATED, {"type": "post_execution"}, causation_id=e_out.event_id)

        return reconstruct_state(self.event_store.get_stream(self.session_id))

print("EventSourcedAgent ready.")


### Executing an Interaction Turn
Let's execute a query and examine the generated immutable event timeline.

In [ ]:
store = EventStore()
agent = EventSourcedAgent(session_id="session-live-01", event_store=store)

# Run read inquiry
final_state = agent.run("Show details for order ORD-101", is_read_only=True)

print(f"Projected State Status: {final_state.status}")
print(f"Final Output: {final_state.final_output}")

print("\n--- Immutable Event Stream ---")
stream = store.get_stream("session-live-01")
for e in stream:
    print(f"Seq #{e.sequence_number:02d} | [{e.event_type.value:<24}] | Hash: {e.metadata.get('hash', '')[:8]}...")


## 6. Time-Travel Debugging & What-If Counterfactual Branching
Because state is an event fold, we can:
- **Time-Travel:** Inspect the agent's exact state at any point in history.
- **Fork Timelines:** Create a branch at sequence $k$ and test counterfactual inputs without altering historical truth.

In [ ]:
# replay_engine.py
class ReplayEngine:
    def __init__(self, event_store: EventStore):
        self.event_store = event_store

    def time_travel_to(self, session_id: str, sequence_number: int) -> AgentState:
        events = self.event_store.get_stream(session_id, from_sequence=1, to_sequence=sequence_number)
        return reconstruct_state(events)

    def fork_stream(self, source_session: str, sequence: int, forked_session: str):
        events = self.event_store.get_stream(source_session, to_sequence=sequence)
        for e in events:
            forked_e = e.model_copy(update={"session_id": forked_session, "metadata": {**e.metadata, "forked": True}})
            self.event_store.append(forked_e)


replay = ReplayEngine(store)

# 1. Time Travel Demonstration
past_state_seq2 = replay.time_travel_to("session-live-01", 2)
print("=== Time-Travel to Sequence 2 (Prompt Received) ===")
print(f"Status: {past_state_seq2.status}")
print(f"Output Available: {past_state_seq2.final_output}")
print(f"Tools Invoked: {len(past_state_seq2.tools_invoked)}")

# 2. What-If Counterfactual Forking
print("\n=== Forking Session at Sequence 2 to Alternate Timeline ===")
replay.fork_stream("session-live-01", sequence=2, forked_session="fork-session-02")

forked_agent = EventSourcedAgent(session_id="fork-session-02", event_store=store)
forked_state = forked_agent.run("Process refund for ORD-101", is_read_only=True)

print(f"Original Session Status: {final_state.status}")
print(f"Forked Session Status (Blocked Policy): {forked_state.status}")
print(f"Forked Session Interceptions: {forked_state.policy_violations}")


## 7. Cryptographic Tamper Detection Verification
Demonstrating that any retroactive mutation to the event log breaks the SHA256 cryptographic hash chain.

In [ ]:
print("1. Verifying uncorrupted stream integrity:")
assert store.verify_stream_integrity("session-live-01") is True
print("   [PASS] Stream hash chain is intact.")

print("\n2. Simulating malicious historical tampering of event #2 payload:")
tampered_stream = store._streams["session-live-01"]
tampered_stream[1] = tampered_stream[1].model_copy(update={"payload": {"prompt": "TAMPERED_INJECTION"}})

try:
    store.verify_stream_integrity("session-live-01")
except EventStoreTamperError as e:
    print(f"   [PASS] Tampering successfully caught by SHA256 chain: {e}")


## 8. Complete Programmatic Test Suite (Automated CI/CD Quality Gate)

In [ ]:
def run_event_sourcing_test_matrix():
    print("===========================================================")
    print("   RUNNING EVENT SOURCING ASSERTION MATRIX                 ")
    print("===========================================================")

    test_store = EventStore()
    test_agent = EventSourcedAgent(session_id="test-ci-01", event_store=test_store)

    # 1. Monotonic Sequencing Assertion
    test_agent.run("Show details for order ORD-101", is_read_only=True)
    stream = test_store.get_stream("test-ci-01")
    seqs = [e.sequence_number for e in stream]
    assert seqs == list(range(1, len(stream) + 1))
    print("[PASS] Monotonic sequencing verified across 8 discrete events.")

    # 2. State Determinism Assertion
    state1 = reconstruct_state(stream)
    state2 = reconstruct_state(stream)
    assert state1.model_dump() == state2.model_dump()
    print("[PASS] State is a pure mathematical fold over events.")

    # 3. Policy Interception Assertion
    sec_agent = EventSourcedAgent(session_id="test-sec-01", event_store=test_store)
    sec_state = sec_agent.run("Process refund for ORD-101", is_read_only=True)
    assert sec_state.status == "HALTED"
    assert sec_state.policy_violations == 1
    print("[PASS] Security policy interception emitted and verified in audit log.")

    # 4. Time Travel Assertion
    test_replay = ReplayEngine(test_store)
    state_at_2 = test_replay.time_travel_to("test-ci-01", 2)
    assert state_at_2.current_sequence == 2
    assert state_at_2.final_output is None
    print("[PASS] Time-travel correctly reconstructed historical checkpoint.")

    print("\n*** ALL EVENT SOURCING ARCHITECTURAL INVARIANTS PASSED! ***")

run_event_sourcing_test_matrix()
